# Data Loading

## Imports

In [1]:
from dotenv import load_dotenv
import numpy as np
import os
import pandas as pd
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values

## Data Processing

In [2]:
DATA_PATH = os.path.join("..", "data")

In [3]:
raw_riders_df = pd.read_csv(os.path.join(DATA_PATH, "riders.csv"))
raw_indiv_results_df = pd.read_csv(os.path.join(DATA_PATH, "individual_results.csv"))
raw_exits_df = pd.read_csv(os.path.join(DATA_PATH, "exits.csv"))

In [4]:
riders_df = raw_riders_df.copy()
indiv_results_df = raw_indiv_results_df.copy()
exits_df = raw_exits_df.copy()

In [5]:
# Convert date columns from str to datetime
riders_df["dob"] = pd.to_datetime(riders_df["dob"])
indiv_results_df["day"] = pd.to_datetime(indiv_results_df["day"])

The rider Aleksandr Vlasov is from Russia, but participated as a neutral contestant. Impute Russia for this row.

In [6]:
riders_df.isna().sum()

bib                   0
rider_name            0
dob                   0
team                  0
rider_country_code    1
rider_country_name    1
team_country_code     0
team_country_name     0
dtype: int64

In [7]:
riders_df[riders_df.isna().any(axis=1)]

,bib,rider_name,dob,team,rider_country_code,rider_country_name,team_country_code,team_country_name
63,78,Aleksandr Vlasov,1996-04-23,RED BULL - BORA - HANSGROHE,NaN,NaN,GER,Germany


In [8]:
riders_df["rider_country_code"] = riders_df["rider_country_code"].fillna("RUS")
riders_df["rider_country_name"] = riders_df["rider_country_name"].fillna("Russian Federation")

In [9]:
riders_df.isna().sum()

bib                   0
rider_name            0
dob                   0
team                  0
rider_country_code    0
rider_country_name    0
team_country_code     0
team_country_name     0
dtype: int64

In [10]:
riders_df.head()

,bib,rider_name,dob,team,rider_country_code,rider_country_name,team_country_code,team_country_name
0,1,Tadej Pogacar,1998-09-21,UAE TEAM EMIRATES XRG,SLO,Slovenia,UAE,United Arab Emirates
1,2,Joao Almeida,1998-08-05,UAE TEAM EMIRATES XRG,POR,Portugal,UAE,United Arab Emirates
2,3,Jhonatan Narvaez,1997-03-04,UAE TEAM EMIRATES XRG,ECU,Ecuador,UAE,United Arab Emirates
3,4,Nils Politt,1994-03-06,UAE TEAM EMIRATES XRG,GER,Germany,UAE,United Arab Emirates
4,5,Pavel Sivakov,1997-07-11,UAE TEAM EMIRATES XRG,FRA,France,UAE,United Arab Emirates


On 18 July 2025, there were 2 contestants with the same rank. Correct the ranks based on [this source](https://www.domestiquecycling.com/en/cycling-races/tour-de-france/2025/stage-13/).

In [11]:
duplicates = indiv_results_df[
    indiv_results_df.duplicated(
        subset=["day", "rank"],
        keep=False
    )
].sort_values(by=["day", "rank"])

display(duplicates)

,day,rider,rank,time,bonus,penalty,start_location,start_country_code,start_country_name,finish_location,finish_country_code,finish_country_name,length,type,name,team
2234,2025-07-18,23,108,1745,0,0,Loudenvielle,FRA,France,Peyragudes,FRA,France,10.9,individual time-trial,Pascal Eenkhoorn,SOUDAL QUICK-STEP
2235,2025-07-18,197,108,1745,0,0,Loudenvielle,FRA,France,Peyragudes,FRA,France,10.9,individual time-trial,Tim Naberman,TEAM PICNIC POSTNL


In [12]:
indiv_results_df.loc[
    (indiv_results_df["day"] == pd.Timestamp("2025-07-18")) &
    (indiv_results_df["rider"] == 197),
    "rank"
] = 109

In [13]:
duplicates = indiv_results_df[
    indiv_results_df.duplicated(
        subset=["day", "rank"],
        keep=False
    )
].sort_values(by=["day", "rank"])

display(duplicates)

,day,rider,rank,time,bonus,penalty,start_location,start_country_code,start_country_name,finish_location,finish_country_code,finish_country_name,length,type,name,team


## Make Tables

In [14]:
riders_table = riders_df[["bib", "rider_name", "dob", "team", "rider_country_code"]]
riders_table = riders_table.rename(columns={
    "bib": "bib_num",
    "rider_name": "name",
    "rider_country_code": "country" 
})

riders_table.head()

,bib_num,name,dob,team,country
0,1,Tadej Pogacar,1998-09-21,UAE TEAM EMIRATES XRG,SLO
1,2,Joao Almeida,1998-08-05,UAE TEAM EMIRATES XRG,POR
2,3,Jhonatan Narvaez,1997-03-04,UAE TEAM EMIRATES XRG,ECU
3,4,Nils Politt,1994-03-06,UAE TEAM EMIRATES XRG,GER
4,5,Pavel Sivakov,1997-07-11,UAE TEAM EMIRATES XRG,FRA


In [15]:
rider_countries_df = riders_df[["rider_country_code", "rider_country_name"]].rename(columns={
    "rider_country_code": "iso",
    "rider_country_name": "name"
})
team_countries_df = riders_df[["team_country_code", "team_country_name"]].rename(columns={
    "team_country_code": "iso",
    "team_country_name": "name"
})
start_countries_df = indiv_results_df[["start_country_code", "start_country_name"]].rename(columns={
    "start_country_code": "iso",
    "start_country_name": "name"
})
finish_countries_df = indiv_results_df[["finish_country_code", "finish_country_name"]].rename(columns={
    "finish_country_code": "iso",
    "finish_country_name": "name"
})
countries_table = pd.concat([
    rider_countries_df,
    team_countries_df,
    start_countries_df,
    finish_countries_df
], ignore_index=True).drop_duplicates().reset_index(drop=True)

countries_table.head()

,iso,name
0,SLO,Slovenia
1,POR,Portugal
2,ECU,Ecuador
3,GER,Germany
4,FRA,France


In [16]:
teams_table = riders_df[["team", "team_country_code"]]
teams_table = teams_table.rename(columns={
    "team": "name",
    "team_country_code": "country" 
}).drop_duplicates().reset_index(drop=True)

teams_table.head()

,name,country
0,UAE TEAM EMIRATES XRG,UAE
1,TEAM VISMA | LEASE A BIKE,NED
2,SOUDAL QUICK-STEP,BEL
3,EF EDUCATION - EASYPOST,USA
4,INTERMARCHÉ - WANTY,BEL


In [17]:
start_locations_df = indiv_results_df[["start_location", "start_country_code"]].rename(columns={
    "start_location": "name",
    "start_country_code": "country"
})
finish_locations_df = indiv_results_df[["finish_location", "finish_country_code"]].rename(columns={
    "finish_location": "name",
    "finish_country_code": "country"
})

locations_table = pd.concat([
    start_locations_df,
    finish_locations_df
], ignore_index=True).drop_duplicates().reset_index(drop=True)

locations_table.head()


,name,country
0,Lille Métropole,FRA
1,Lauwin-Planque,FRA
2,Valenciennes,FRA
3,Amiens Métropole,FRA
4,Caen,FRA


In [18]:
stages_table = indiv_results_df[["day", "start_location", "finish_location", "length", "type"]] \
    .rename(columns={
        "day": "date",
        "finish_location": "end_location",
        "length": "dist_covered",
    }) \
    .drop_duplicates() \
    .sort_values(by="date", ascending=True) \
    .reset_index(drop=True)

# Add stage id
stages_table.insert(0, "id", np.arange(1, len(stages_table) + 1))

stages_table.head()

,id,date,start_location,end_location,dist_covered,type
0,1,2025-07-05,Lille Métropole,Lille Métropole,184.9,flat
1,2,2025-07-06,Lauwin-Planque,Boulogne-Sur-Mer,209.1,hilly
2,3,2025-07-07,Valenciennes,Dunkerque,178.3,flat
3,4,2025-07-08,Amiens Métropole,Rouen,174.2,hilly
4,5,2025-07-09,Caen,Caen,33.0,individual time-trial


In [19]:
stage_details_to_ids = {
    (
        row.date,
        row.start_location,
        row.end_location,
        row.dist_covered,
        row.type
    ): row.id
    for row in stages_table.itertuples(index=False)
}

stage_details_to_ids

{(Timestamp('2025-07-05 00:00:00'),
  'Lille Métropole',
  'Lille Métropole',
  184.9,
  'flat'): 1,
 (Timestamp('2025-07-06 00:00:00'),
  'Lauwin-Planque',
  'Boulogne-Sur-Mer',
  209.1,
  'hilly'): 2,
 (Timestamp('2025-07-07 00:00:00'),
  'Valenciennes',
  'Dunkerque',
  178.3,
  'flat'): 3,
 (Timestamp('2025-07-08 00:00:00'),
  'Amiens Métropole',
  'Rouen',
  174.2,
  'hilly'): 4,
 (Timestamp('2025-07-09 00:00:00'),
  'Caen',
  'Caen',
  33.0,
  'individual time-trial'): 5,
 (Timestamp('2025-07-10 00:00:00'),
  'Bayeux',
  'Vire Normandie',
  201.5,
  'hilly'): 6,
 (Timestamp('2025-07-11 00:00:00'),
  'Saint-Malo',
  'Mûr-De-Bretagne Guerlédan',
  197.0,
  'hilly'): 7,
 (Timestamp('2025-07-12 00:00:00'),
  'Saint-Méen-Le-Grand',
  'Laval Espace Mayenne',
  171.4,
  'flat'): 8,
 (Timestamp('2025-07-13 00:00:00'), 'Chinon', 'Châteauroux', 174.1, 'flat'): 9,
 (Timestamp('2025-07-14 00:00:00'),
  'Ennezat',
  'Le Mont-Dore Puy De Sancy',
  165.3,
  'mountain'): 10,
 (Timestamp('2025-07

In [20]:
stage_results_table = indiv_results_df.copy()
stage_results_table["stage_id"] = stage_results_table.apply(
    lambda row: stage_details_to_ids[(
        row["day"],
        row["start_location"],
        row["finish_location"],
        row["length"],
        row["type"]
    )], axis=1
)
stage_results_table = stage_results_table[["rider", "stage_id", "rank", "time", "bonus", "penalty"]] \
    .rename(columns={
        "rider": "rider_bib",
        "time": "finish_time",
        "bonus": "time_bonus",
        "penalty": "time_penalty"
    })

stage_results_table.head()

,rider_bib,stage_id,rank,finish_time,time_bonus,time_penalty
0,101,1,1,13991,10,0
1,41,1,2,13991,6,0
2,228,1,3,13991,4,0
3,187,1,4,13991,0,0
4,118,1,5,13991,0,0


In [21]:
rider_exits_table = exits_df[["bib", "stage", "reason"]] \
    .rename(columns={
        "bib": "rider_bib",
        "stage": "stage_id"
    })

rider_exits_table.head()

,rider_bib,stage_id,reason
0,163,14,DNS
1,66,18,DNS
2,173,12,DNS
3,93,18,DNS
4,77,17,DNS


In [22]:
awards_table = pd.DataFrame({
    "name": [
        "general",
        "points",
        "king of the mountains",
        "young rider",
        "best team"
    ]
})

## Verify Table Correctness

### Uniqueness Constraints

In [23]:
# Primary key uniqueness
assert countries_table["iso"].is_unique
assert teams_table["name"].is_unique
assert riders_table["bib_num"].is_unique
assert locations_table["name"].is_unique
assert stages_table["id"].is_unique
assert rider_exits_table["rider_bib"].is_unique
assert awards_table["name"].is_unique

# Composite primary key uniqueness
assert not stage_results_table.duplicated(
    ["rider_bib", "stage_id"]
).any()

# Additional unique constraint should be obeyed
assert not stage_results_table.duplicated(
    ["stage_id", "rank"]
).any()

### NULL Values

In [24]:
# There should not be any NULL values
table_names_to_tables = {
    "countries": countries_table,
    "teams": teams_table,
    "riders": riders_table,
    "locations": locations_table,
    "stages": stages_table,
    "stage_results": stage_results_table,
    "rider_exits": rider_exits_table,
    "awards": awards_table
}

for table_name, table in table_names_to_tables.items():
    assert not table.isna().any().any(), f"Missing values in {table_name}"

### Domain Constraints

In [25]:
assert (riders_table["dob"] < pd.Timestamp.today()).all()

assert (stages_table["dist_covered"] > 0).all()

assert stages_table["type"].isin([
    "flat",
    "hilly",
    "mountain",
    "individual time-trial"
]).all()

assert (stage_results_table["rank"] >= 1).all()

for col in ["finish_time", "time_bonus", "time_penalty"]:
    assert (stage_results_table[col] >= 0).all()

assert countries_table["iso"].str.len().eq(3).all()

### Foreign Key Constraints

In [26]:
assert teams_table["country"].isin(countries_table["iso"]).all()

assert riders_table["team"].isin(teams_table["name"]).all()
assert riders_table["country"].isin(countries_table["iso"]).all()

assert locations_table["country"].isin(countries_table["iso"]).all()

assert stages_table["start_location"].isin(locations_table["name"]).all()
assert stages_table["end_location"].isin(locations_table["name"]).all()

assert stage_results_table["rider_bib"].isin(riders_table["bib_num"]).all()
assert stage_results_table["stage_id"].isin(stages_table["id"]).all()

assert rider_exits_table["rider_bib"].isin(riders_table["bib_num"]).all()
assert rider_exits_table["stage_id"].isin(stages_table["id"]).all()

### Participation Constraints

In [27]:
# Every team has at least one rider
assert teams_table["name"].isin(riders_table["team"]).all()

# Every location must appear as the start or end of at least one stage
used_locations = pd.concat([
    stages_table["start_location"],
    stages_table["end_location"]
])

assert locations_table["name"].isin(used_locations).all()

### Rider-Exit Constraint

In [28]:
# If a rider exits before stage S, that rider cannot have results for stage S or any subsequent stage
results_with_exits = stage_results_table.merge(
    rider_exits_table,
    on="rider_bib",
    how="inner",
    suffixes=("_result", "_exit")
)

assert (
    results_with_exits["stage_id_result"]
    < results_with_exits["stage_id_exit"]
).all()

### Reconstruct Original Tables To Verify Correctness

In [29]:
def assert_same_rows(df1, df2):
    assert list(df1.columns) == list(df2.columns)

    pd.testing.assert_frame_equal(
        df1.sort_values(
            by=list(df1.columns)
        ).reset_index(drop=True),
        df2.sort_values(
            by=list(df2.columns)
        ).reset_index(drop=True),
        check_dtype=False
    )

In [30]:
reconstructed_riders = (
    riders_table
    .merge(
        teams_table.rename(columns={
            "name": "team",
            "country": "team_country_code"
        }),
        on="team",
        validate="many_to_one"
    )
    .merge(
        countries_table.rename(columns={
            "iso": "rider_country_code",
            "name": "rider_country_name"
        }),
        left_on="country",
        right_on="rider_country_code",
        validate="many_to_one"
    )
    .merge(
        countries_table.rename(columns={
            "iso": "team_country_code",
            "name": "team_country_name"
        }),
        on="team_country_code",
        validate="many_to_one"
    )
    .rename(columns={
        "bib_num": "bib",
        "name": "rider_name"
    })
)

columns = [
    "bib",
    "rider_name",
    "dob",
    "team",
    "rider_country_code",
    "rider_country_name",
    "team_country_code",
    "team_country_name"
]

assert_same_rows(
    riders_df[columns],
    reconstructed_riders[columns]
)

In [31]:
reconstructed_results = (
    stage_results_table
    .merge(
        stages_table,
        left_on="stage_id",
        right_on="id",
        validate="many_to_one"
    )
    .merge(
        locations_table.rename(columns={
            "name": "start_location",
            "country": "start_country_code"
        }),
        on="start_location",
        validate="many_to_one"
    )
    .merge(
        locations_table.rename(columns={
            "name": "end_location",
            "country": "finish_country_code"
        }),
        on="end_location",
        validate="many_to_one"
    )
    .merge(
        countries_table.rename(columns={
            "iso": "start_country_code",
            "name": "start_country_name"
        }),
        on="start_country_code",
        validate="many_to_one"
    )
    .merge(
        countries_table.rename(columns={
            "iso": "finish_country_code",
            "name": "finish_country_name"
        }),
        on="finish_country_code",
        validate="many_to_one"
    )
    .merge(
        riders_table[["bib_num", "name", "team"]],
        left_on="rider_bib",
        right_on="bib_num",
        validate="many_to_one"
    )
    .rename(columns={
        "rider_bib": "rider",
        "date": "day",
        "end_location": "finish_location",
        "dist_covered": "length",
        "finish_time": "time",
        "time_bonus": "bonus",
        "time_penalty": "penalty"
    })
)

# Match the original DataFrame's column order
reconstructed_results = reconstructed_results[indiv_results_df.columns]

assert_same_rows(
    indiv_results_df,
    reconstructed_results
)

## Insert Into Database

In [32]:
def insert_dataframe(cursor, table_name, df):
    columns = list(df.columns)

    query = sql.SQL("INSERT INTO {} ({}) VALUES %s").format(
        sql.Identifier(table_name),
        sql.SQL(", ").join(map(sql.Identifier, columns))
    )

    rows = list(df.itertuples(index=False, name=None))

    execute_values(cursor, query.as_string(cursor), rows)

In [33]:
load_dotenv()

db_config = {
    "host": os.environ.get("DB_HOST"),
    "port": os.environ.get("DB_PORT"),
    "database": os.environ.get("DB_NAME"),
    "user": os.environ.get("DB_USER"),
    "password": os.environ.get("DB_PASSWORD")
}

In [34]:
tables_to_insert = [
    ("countries", countries_table),
    ("teams", teams_table),
    ("riders", riders_table),
    ("locations", locations_table),
    ("stages", stages_table),
    ("stage_results", stage_results_table),
    ("rider_exits", rider_exits_table),
    ("awards", awards_table)
]

In [35]:
try:
    with psycopg2.connect(**db_config) as connection:
        with connection.cursor() as cursor:
            for table_name, df in tables_to_insert:
                insert_dataframe(cursor, table_name, df)
                print(f"Inserted {len(df)} rows into {table_name}")

    print("All tables loaded successfully.")

except Exception as error:
    print(f"Database error: {error}")

Inserted 30 rows into countries
Inserted 23 rows into teams
Inserted 184 rows into riders
Inserted 39 rows into locations
Inserted 21 rows into stages
Inserted 3602 rows into stage_results
Inserted 24 rows into rider_exits
Inserted 5 rows into awards
All tables loaded successfully.
